# extractrs vs exactextract — Validation

This notebook example validates that extractrs produces **bit-identical** results as `exactextract` on all ~225K CONUS MERIT Hydro basins.

- **Raster**: Daymet V4R1 daily tmin (local NetCDF)
- **Basins**: MERIT Hydro Pfafstetter level-7 basins (local shapefile)

Citations:
Thornton, M. M., Shrestha, R., Wei, Y., Thornton, P. E., Kao, S.-C., & Wilson, B. E. (2022). Daymet: Daily Surface Weather Data on a 1-km Grid for North America, Version 4 R1 (Version 4.4) [netCDF]. ORNL Distributed Active Archive Center. https://doi.org/10.3334/ORNLDAAC/2129

Lin, P., Pan, M., Wood, E. F., Yamazaki, D., & Allen, G. H. (2021). A new vector-based global river network dataset accounting for variable drainage density. Scientific Data, 8(1), 28. https://doi.org/10.1038/s41597-021-00819-9

In [1]:
!uv pip install extractrs[rio] exactextract

Using Python 3.13.12 environment at: /home/tbindas/projects/extractrs/.venv
Audited 2 packages in 2ms


In [2]:
import os
import tempfile
import time
import warnings

import exactextract
import extractrs as extrs
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import rioxarray  # noqa: F401 — registers .rio accessor
import xarray as xr
from rasterio.transform import from_origin
from rasterio.windows import from_bounds

## Configuration

In [3]:
NC_PATH = "/mnt/ssd1/data/daymet/daymet_v4_daily_na_tmin_2024.nc"
SHP_PATH = "/mnt/ssd1/data/merit/cat_pfaf_7_MERIT_Hydro_v07_Basins_v01_bugfix1.shp"
VAR = "tmin"
TIME_IDX = 182  # July 1 (0-indexed)
ID_COL = "COMID"
N_BENCH = 10

# Full CONUS
CONUS_BBOX = (-125.0, 24.0, -66.0, 50.0)

BAD_COMIDS = {74009117}  # known to segfault exactextract

DAYMET_CRS = (
    "+proj=lcc +lat_0=42.5 +lon_0=-100 +lat_1=25 +lat_2=60 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m"
)

## Load Daymet

In [4]:
ds = xr.open_dataset(NC_PATH)
ds = ds[[VAR]].isel(time=TIME_IDX)
ds = ds.rio.write_crs(DAYMET_CRS)

da = ds[VAR]
x = da["x"].values.astype(np.float64)
y = da["y"].values.astype(np.float64)
dx, dy = abs(float(x[1] - x[0])), abs(float(y[1] - y[0]))
full_transform = from_origin(float(x.min()) - dx / 2, float(y.max()) + dy / 2, dx, dy)

print(f"Dataset: {dict(ds.sizes)}, resolution: {dx}x{dy}m")

Dataset: {'y': 8075, 'x': 7814}, resolution: 1000.0x1000.0m


## Benchmark — Full CONUS (~225K basins)

Load all MERIT Hydro basins in the lower 48 and benchmark both tools at scale.

In [5]:
t0 = time.time()
conus_basins = gpd.read_file(SHP_PATH, bbox=CONUS_BBOX)
conus_basins = conus_basins.set_crs("EPSG:4326", allow_override=True)
conus_basins = conus_basins[~conus_basins[ID_COL].isin(BAD_COMIDS)]
conus_proj = conus_basins.to_crs(DAYMET_CRS)
print(f"{len(conus_basins):,} CONUS basins loaded in {time.time()-t0:.1f}s")

225,063 CONUS basins loaded in 11.2s


In [6]:
# Build GeoTIFF covering full CONUS for exactextract
tb = conus_proj.total_bounds
pad = 2000
window = from_bounds(tb[0] - pad, tb[1] - pad, tb[2] + pad, tb[3] + pad, full_transform)
window = window.round_offsets().round_lengths(op="ceil")

row_off, col_off = int(window.row_off), int(window.col_off)
height, width = int(window.height), int(window.width)

data_conus = da.values[row_off:row_off + height, col_off:col_off + width]
conus_win_transform = rasterio.windows.transform(window, full_transform)

conus_tif = os.path.join(tempfile.mkdtemp(), "daymet_conus.tif")
with rasterio.open(
    conus_tif, "w", driver="GTiff",
    height=height, width=width, count=1,
    dtype=data_conus.dtype, crs=DAYMET_CRS,
    transform=conus_win_transform, nodata=-9999.0,
) as dst:
    dst.write(data_conus, 1)

print(f"CONUS GeoTIFF: {width}x{height}")

CONUS GeoTIFF: 4525x3280


In [7]:
# exactextract CONUS benchmark
N_CONUS_BENCH = 3

rast_conus = rasterio.open(conus_tif)
ee_times = []
for i in range(N_CONUS_BENCH):
    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        exactextract.exact_extract(rast_conus, conus_proj, ops=["mean"],
                                   include_cols=[ID_COL], output="pandas")
    elapsed = time.time() - t0
    ee_times.append(elapsed)
    print(f"  exactextract run {i+1}: {elapsed:.1f}s")
rast_conus.close()

ee_median = np.median(ee_times)
print(f"  median: {ee_median:.1f}s")

  exactextract run 1: 137.7s
  exactextract run 2: 137.3s
  exactextract run 3: 137.3s
  median: 137.3s


In [8]:
# extractrs CONUS benchmark
data_2d = ds[VAR].values.astype(np.float64)
nodata = float(ds[VAR].encoding.get("_FillValue", -9999.0))

t0 = time.time()
cache = extrs.build_cache(
    [g.wkb for g in conus_proj.geometry],
    conus_proj[ID_COL].astype(np.int64).tolist(),
    float(x.min()) - dx / 2, float(y.min()) - dy / 2,
    float(x.max()) + dx / 2, float(y.max()) + dy / 2, dx, dy,
)
cache_time = time.time() - t0
print(f"  cache build: {cache_time:.1f}s ({cache.n_basins:,} basins)")

rs_times = []
for i in range(N_BENCH):
    t0 = time.time()
    extrs.apply_stat(cache, data_2d, nodata, "mean")
    elapsed = time.time() - t0
    rs_times.append(elapsed)

rs_median = np.median(rs_times)
print(f"  apply_stat median: {rs_median:.3f}s ({N_BENCH} runs)")

print(f"\n{'='*50}")
print(f"CONUS BENCHMARK ({len(conus_basins):,} basins)")
print(f"{'='*50}")
print(f"exactextract:  {ee_median:.1f}s/run")
print(f"extractrs:")
print(f"  cache build: {cache_time:.1f}s (one-time)")
print(f"  apply_stat:  {rs_median:.3f}s/run")
if rs_median > 0:
    print(f"  speedup:     {ee_median/rs_median:.0f}x")

  cache build: 8.0s (225,063 basins)
  apply_stat median: 0.150s (10 runs)

CONUS BENCHMARK (225,063 basins)
exactextract:  137.3s/run
extractrs:
  cache build: 8.0s (one-time)
  apply_stat:  0.150s/run
  speedup:     914x
